In [ ]:
Part 1: Read Data
Tasks:

Read the dataset Q3_data.csv using read_csv()

Inspect the first few rows using head()

Display dataset information using info()

Show statistical description using describe()


Part 2: Data Cleaning
Tasks:

Inspect and fix the following when needed:

Handle missing values appropriately

Check and remove duplicates if any exist

Encode categorical variables if needed

Apply feature scaling to numerical features (Use StandardScaler)

Check for target imbalance and state if it is imbalanced or not


Part 3: Modeling
Tasks:

Split the dataset into features (X) and target (y)

Use the correct split: KFold OR StratifiedKFold

Train a CatBoostClassifier model

Evaluate using the appropriate metric only (Accuracy vs. F1 Score).

Print the averaged score across all folds


Part 4: Find the Golden Feature!
Tasks:

Plot feature importance from your trained model

Identify and print the name of the most important feature (the 'golden feature')



Part 5: Bonus - Retrain with Golden Feature Only
Task:

Now that you've found the golden feature, let's see how powerful it really is!

Retrain your CatBoostClassifier using ONLY the golden feature and compare its performance to the full model:

Create new X with only the golden feature

Run the same KFold loop with this single feature

Print and compare the accuracy with the full model

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score


In [ ]:
# Task 1: Write your code here:
Q3_data_path = os.path.join(path, 'Q3_data.csv')
df= pd.read_csv(Q3_data_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Analyze missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
df = df.dropna()
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Task 2: Write your code here:
# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

In [ ]:
df.describe()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # <Replace None with your code>

scaler = StandardScaler()

# TODO: Apply fit_transform to scale the numerical columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])  # <Replace None with your code>

df.head()


In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, test_idx in kf.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier(verbose=0, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    scores.append(f1_score(y_test, y_pred))

print("Average F1 Score:", np.mean(scores))


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

model.fit(X_scaled, y)

importances = model.get_feature_importance()
feature_names = X.columns

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(10,6))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.gca().invert_yaxis()
plt.show()



In [ ]:
# Task 2: Write your code here:
golden_feature = importance_df.iloc[0]["feature"]
print("Golden Feature:", golden_feature)

In [ ]:
# Task Bonus: Write your code here:
X_golden = df[[golden_feature]]

X_golden_scaled = scaler.fit_transform(X_golden)

scores_golden = []

for train_idx, test_idx in kf.split(X_golden_scaled, y):
    X_train, X_test = X_golden_scaled[train_idx], X_golden_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier(verbose=0, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    scores_golden.append(f1_score(y_test, y_pred))

print("Golden Feature F1 Score:", np.mean(scores_golden))